# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR² rangeland management practices dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

The dataset uses unique `@id` values to reference all record sets, fields, and columns, as per the Croissant standard.

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {getattr(metadata, 'version', 'N/A')}, Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review and display available record sets and their fields, by their `@id` values.

In [ ]:
# List all record sets. Display their @id and contained fields (by @id).
print("Record sets available in this dataset:\n")
rs_ids = []
for rs in dataset.record_sets:
    print(f"- Record Set @id: {rs.id}, name: {getattr(rs, 'name', '(no name)')}")
    rs_ids.append(rs.id)
    if hasattr(rs, 'fields'):
        for f in rs.fields:
            print(f"    - Field @id: {f.id}, name: {getattr(f, 'name', '(unknown)')}")
    print()
# Save the list for later
record_set_ids = rs_ids

## 3. Data Extraction
Load data from each record set into Pandas DataFrames for analysis. All references use the record set and field `@id` values.

In [ ]:
# Extract data from each record set into a DataFrame, using the @id
dataframes = {}

for record_set_id in record_set_ids:
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet @id: {record_set_id}, shape: {df.shape}")
    if not df.empty:
        print(f"  Columns: {list(df.columns)}\n")

# For demonstration, pick the first non-empty record set for further analysis.
main_record_set_id = None
for record_set_id in record_set_ids:
    if not dataframes[record_set_id].empty:
        main_record_set_id = record_set_id
        break
if main_record_set_id is None:
    raise ValueError('No non-empty record set found in the dataset!')
else:
    print(f"Using RecordSet @id: {main_record_set_id} for sample EDA.")
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps:
- Filtering records by a numeric field
- Normalizing numeric fields
- Grouping by key fields

_All field and record set references use their Croissant `@id`s for reproducibility._

In [ ]:
# Select a numeric field (by @id) for demonstration.
# We will look up field types using the record set definitions.

rs_obj = None
for rs in dataset.record_sets:
    if rs.id == main_record_set_id:
        rs_obj = rs
        break

# Find numeric fields (those with dataType Float or Integer)
numeric_fields = []
if rs_obj and hasattr(rs_obj, 'fields'):
    for f in rs_obj.fields:
        if hasattr(f, 'data_type') and f.data_type is not None:
            dt = str(f.data_type).lower()
            if 'float' in dt or 'integer' in dt or 'number' in dt:
                numeric_fields.append(f.id)

if not numeric_fields:
    print('No numeric fields found in selected record set.')
else:
    print(f"Numeric fields (@id): {numeric_fields}")
    numeric_field_id = numeric_fields[0]
    df = dataframes[main_record_set_id]
    # Cast to numeric (handle missing values)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records from RecordSet '{main_record_set_id}' with field '{numeric_field_id}' > {threshold:.2f} (mean):")
    print(filtered_df.head())

    # Normalize
    if filtered_df[numeric_field_id].std() != 0:
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical/text field:
    group_field = None
    for f in rs_obj.fields:
        if hasattr(f, 'data_type') and f.data_type is not None:
            dt = str(f.data_type).lower()
            if 'text' in dt or 'string' in dt:
                group_field = f.id
                break
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"\nGrouped by field: {group_field}")
        print(grouped_df.head())
    else:
        print("\nNo suitable categorical/text field for grouping found.")
else:
    print('Skip EDA: no numeric field detected in record set.')

## 5. Visualization
Visualize the distribution of the selected numeric field from the above analysis. This example uses matplotlib for plotting.

In [ ]:
# Plot histogram for the selected numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if not numeric_fields:
    print('No numeric fields available for plotting.')
else:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of field {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

## 6. Conclusion

- This notebook demonstrated programmatic exploration of a Croissant dataset using record set and field `@id` references as recommended by the [mlcroissant](https://github.com/mlcommons/croissant) model.
- We loaded all available record sets, explored the schema, and analyzed a numeric field with visualization.
- You should further adapt this notebook to your own analytical and modeling needs, always referring to entities by their Croissant `@id` for reproducibility and transparency.
